In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

from starter_pack.code.align_images  import *
from starter_pack.code.utils  import *
# print(align_images)
# print(type(align_images))

In [6]:
def gaussian_1d(sigma):
    r = int(np.ceil(3 * sigma))
    x = np.arange(-r, r + 1)
    g = np.exp(-x ** 2 / (2 * sigma ** 2))
    return g / g.sum()

In [7]:
def conv1d_reflect(img, k, axis):
    """Convolve a 2D image with 1D kernel k along `axis`, reflect padding, same size."""
    r = len(k) // 2
    pad = [(0, 0), (0, 0)]
    pad[axis] = (r, r)
    p = np.pad(img, pad, mode="reflect")
    k = k[::-1]                      # convolution = correlation with flipped kernel
    out = np.zeros_like(img, dtype=np.float64)
    n = img.shape[axis]
    for i, w in enumerate(k):        # loop over kernel taps only, whole-image ops
        sl = [slice(None), slice(None)]
        sl[axis] = slice(i, i + n)
        out += w * p[tuple(sl)]
    return out


In [8]:
def blur(img, sigma):
    """Separable Gaussian blur on a gray (H,W) or color (H,W,C) image, per channel."""
    k = gaussian_1d(sigma)
    if img.ndim == 2:
        return conv1d_reflect(conv1d_reflect(img, k, 0), k, 1)
    return np.stack([blur(img[..., c], sigma) for c in range(img.shape[2])], axis=2)

In [9]:
def cutoff_cpi(sigma, width):
    """Frequency (cycles/image) where a Gaussian's gain = 1/2 (the paper's definition)."""
    return np.sqrt(np.log(2) / 2) / (np.pi * sigma) * width
 
 
def make_hybrid(im_far, im_near, sigma_low, sigma_high):
    low = blur(im_far, sigma_low)                 # seen from far away
    high = im_near - blur(im_near, sigma_high)    # seen up close
    hybrid = np.clip(low + high, 0, 1)
    return low, high, hybrid

In [10]:
def log_fft(img):
    gray = img if img.ndim == 2 else cv2.cvtColor(img.astype(np.float32), cv2.COLOR_RGB2GRAY)
    return np.log(np.abs(np.fft.fftshift(np.fft.fft2(gray))) + 1e-8)

In [11]:
def fit_size(im, max_side=1000):
    """Bring an image to <= max_side on its longer side. Pick align points AFTER this."""
    s = max_side / max(im.shape[:2])
    if s >= 1:
        return im
    return cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)

In [12]:
def run(name, path_far, path_near, sigma_low, sigma_high,
        pts_far=None, pts_near=None, gray=True):
    """pts_far / pts_near: two (x, y) points in each image (e.g. the two eyes).
    If None, you click them (pick_points; local Python only)."""
    im1 = fit_size(load_image(path_far, gray=gray))
    im2 = fit_size(load_image(path_near, gray=gray))
 
    if pts_far is None:
        pts_far = pick_points(im1)
    if pts_near is None:
        pts_near = pick_points(im2)
    print(f"[{name}] pts_far={pts_far}  pts_near={pts_near}")   # copy these into the call to reuse
 
    im2_al = align_images(im1, im2, pts_far, pts_near)          # im2 warped onto im1
 
    low, high, hybrid = make_hybrid(im1, im2_al, sigma_low, sigma_high)
 
    W = hybrid.shape[1]
    print(f"[{name}] sigma_low={sigma_low} (fc={cutoff_cpi(sigma_low, W):.1f} c/i), "
          f"sigma_high={sigma_high} (fc={cutoff_cpi(sigma_high, W):.1f} c/i)")
 
    # ---- process figure: images on top, log-magnitude FFT below ----
    imgs = [im1, im2_al, low, high + 0.5, hybrid]               # +0.5: zero -> mid-gray
    titles = ["Image 1 (low-pass source)", "Image 2, aligned", f"Low-pass σ={sigma_low}",
              f"High-pass σ={sigma_high}", "Hybrid"]
    ffts = [log_fft(im1), log_fft(im2_al), log_fft(low), log_fft(high), log_fft(hybrid)]
 
    fig, ax = plt.subplots(2, 5, figsize=(20, 8))
    for i in range(5):
        shown = np.clip(imgs[i], 0, 1)
        ax[0, i].imshow(shown, cmap="gray" if shown.ndim == 2 else None, vmin=0, vmax=1)
        ax[0, i].set_title(titles[i])
        ax[0, i].axis("off")
        ax[1, i].imshow(ffts[i], cmap="gray")
        ax[1, i].set_title("log |FFT|")
        ax[1, i].axis("off")
    plt.tight_layout()
    plt.savefig(f"{name}_process.png", dpi=120)
    plt.show()
 
    save_image(f"{name}_hybrid.png", hybrid)
    small = cv2.resize(hybrid.astype(np.float32), None, fx=0.2, fy=0.2, interpolation=cv2.INTER_AREA)
    save_image(f"{name}_hybrid_small.png", small)               # simulates viewing from far away
    return hybrid

In [20]:
run(
    "man_cat",
    "starter_pack/images/part2_frequencies/hybrid_man.jpg",
    "starter_pack/images/part2_frequencies/hybrid_cat.jpg",
    sigma_low=2,
    sigma_high=2
)

[man_cat] pts_far=[(392.71428571428555, 333.91558441558436), (326.15584415584397, 342.0324675324674)]  pts_near=[(501.11290322580646, 266.4354838709678), (441.4354838709677, 224.5000000000001)]
[man_cat] sigma_low=2 (fc=67.0 c/i), sigma_high=2 (fc=67.0 c/i)


array([[0.39014739, 0.314667  , 0.26037862, ..., 0.40212196, 0.40280012,
        0.41385462],
       [0.45633583, 0.35737231, 0.29692773, ..., 0.39836694, 0.40710815,
        0.41498949],
       [0.49841696, 0.38281388, 0.32004314, ..., 0.41000788, 0.41554461,
        0.41821528],
       ...,
       [0.01614268, 0.00775558, 0.0129336 , ..., 0.12694359, 0.12952285,
        0.14232777],
       [0.0191897 , 0.01883443, 0.02485881, ..., 0.14167016, 0.13325809,
        0.13568944],
       [0.0229035 , 0.0281044 , 0.04495378, ..., 0.1257148 , 0.11882215,
        0.11429142]], shape=(1000, 715))

In [14]:
# %matplotlib qt